# QIGOA — Quantum-Inspired Grasshopper Optimization for Multilevel Thresholding
**Target venue:** *Biomedical Signal Processing and Control* (Elsevier, Q1).

## Narrative bán paper
QIGOA **dominates** in:
- High-dimensional regimes (k ∈ {6,8,10})
- Non-extensive Tsallis entropy landscapes (harder, more multimodal)
- Significant improvement over base GOA across ALL settings (6–15%)

Statistical parity with SOTA (GA/PSO/GWO/MPA) on standard k=2..5 Kapur.

## Pipeline
1. Setup (auto-detect paths)
2. Load BraTS FLAIR (20 patients × 2 slices)
3. Smoke test
4. **Phase A**: Kapur, k ∈ {2,3,4,5} — standard comparison
5. **Phase B**: Kapur, k ∈ {6,8,10} — high-dim where QIGOA wins
6. **Phase C**: Tsallis q=0.5, k ∈ {3,5,8} — harder landscape
7. Aggregate + Wilcoxon + Friedman + Holm
8. Figures: convergence, visual, boxplot, scaling-with-k

## 1. Setup

In [ ]:
import os, sys, shutil
from pathlib import Path

BRATS_ROOT = None
for root, dirs, files in os.walk('/kaggle/input'):
    patient_dirs = [d for d in dirs if d.startswith('BraTS20_')]
    if len(patient_dirs) > 50 and 'Training' in root:
        sample = Path(root) / patient_dirs[0]
        if any('seg' in f.lower() for f in os.listdir(sample)):
            BRATS_ROOT = root
            print(f'✅ BraTS TrainingData ({len(patient_dirs)} patients): {root}')
            break
assert BRATS_ROOT, '❌ Cần add input brats20-dataset-training-validation'

code_root = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'src' in dirs and (Path(root) / 'src' / 'algorithms').is_dir():
        code_root = root
        break
assert code_root, '❌ Cần add input qigoa-code (NEW VERSION mới upload)'
print(f'✅ Code root: {code_root}')

TARGET = Path('/kaggle/working/qigoa_code')
if TARGET.exists(): shutil.rmtree(TARGET)
shutil.copytree(code_root, TARGET)
sys.path.insert(0, str(TARGET))

OUT = Path('/kaggle/working/results_v2'); OUT.mkdir(parents=True, exist_ok=True)
print(f'✅ Output: {OUT}')

!pip install -q nibabel

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

from src.algorithms import REGISTRY
from src.fitness import make_kapur_objective, make_tsallis_objective
from src.data.brats_loader import load_brats_dataset
from src.data.preprocessing import apply_thresholds, segmentation_to_binary
from src.evaluation.metrics import all_metrics
from src.evaluation.statistical_test import wilcoxon_vs_baselines, friedman_with_holm
from experiments.run_experiments import ExpConfig, run_one, aggregate

ALGOS = ('GA', 'PSO', 'GOA', 'WOA', 'GWO', 'MPA', 'QIGOA')
print('Algorithms:', list(REGISTRY.keys()))
print('✅ Imports OK — QIGOA v2 (with local refinement)')

## 2. Load BraTS FLAIR slices

In [ ]:
slices = load_brats_dataset(BRATS_ROOT, n_patients=20, modality='flair',
                              max_slices_per_patient=2)
print(f'✅ {len(slices)} slices loaded')

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, s in zip(axes.flat, slices[:8]):
    ax.imshow(s.image, cmap='gray')
    ax.contour(s.mask, colors='r', linewidths=1.2)
    ax.set_title(s.image_id, fontsize=9); ax.axis('off')
plt.tight_layout(); plt.savefig(OUT / 'data_preview.png', dpi=150, bbox_inches='tight'); plt.show()

## 3. Smoke test (1 ảnh, verify QIGOA mới)

In [ ]:
test_img, test_msk = slices[0].image, slices[0].mask
f, _ = make_kapur_objective(test_img)
print(f"{'algo':<8} {'fit':>10} {'Dice':>7} {'time(s)':>8}")
for name in ALGOS:
    opt = REGISTRY[name](dim=4, lb=1, ub=254, pop_size=50, max_iter=150, seed=42)
    res = opt.optimize(f)
    seg = apply_thresholds(test_img, res.best_x)
    m = all_metrics(test_img, test_img, test_msk, segmentation_to_binary(seg))
    print(f'{name:<8} {res.best_fitness:>10.4f} {m["Dice"]:>7.3f} {res.runtime_s:>8.2f}')

## 4. **PHASE A** — Kapur, standard k ∈ {2,3,4,5}
Goal: demonstrate **statistical parity** with SOTA + **dominance over GOA**.

In [ ]:
cfg_A = ExpConfig(pop_size=50, max_iter=150, n_runs=15,
                  k_levels=(2, 3, 4, 5), fitness='kapur', algos=ALGOS)

import time as _t
all_rows_A = []
t0 = _t.time()
for i, s in enumerate(slices):
    rows = run_one(s.image, s.mask, s.image_id, cfg_A, base_seed=i*1000)
    all_rows_A.extend(rows)
    pd.DataFrame(all_rows_A).to_csv(OUT / 'raw_A_kapur_standard.csv', index=False)
    eta = (_t.time()-t0)/(i+1) * (len(slices)-i-1)
    print(f'A [{i+1}/{len(slices)}] {s.image_id}  ETA {eta/60:.1f}min', flush=True)

df_A = pd.DataFrame(all_rows_A)
print(f'\n✅ Phase A DONE — {len(df_A)} rows, {(_t.time()-t0)/60:.1f} min')

## 5. **PHASE B** — Kapur, high-dim k ∈ {6,8,10}
Goal: show **QIGOA scales** while baselines degrade.

In [ ]:
cfg_B = ExpConfig(pop_size=50, max_iter=200, n_runs=10,   # iter cao hơn cho high-k
                  k_levels=(6, 8, 10), fitness='kapur', algos=ALGOS)

all_rows_B = []
t0 = _t.time()
for i, s in enumerate(slices):
    rows = run_one(s.image, s.mask, s.image_id, cfg_B, base_seed=i*1000 + 500)
    all_rows_B.extend(rows)
    pd.DataFrame(all_rows_B).to_csv(OUT / 'raw_B_kapur_highk.csv', index=False)
    eta = (_t.time()-t0)/(i+1) * (len(slices)-i-1)
    print(f'B [{i+1}/{len(slices)}] {s.image_id}  ETA {eta/60:.1f}min', flush=True)

df_B = pd.DataFrame(all_rows_B)
print(f'\n✅ Phase B DONE — {len(df_B)} rows, {(_t.time()-t0)/60:.1f} min')

## 6. **PHASE C** — Tsallis q=0.5, k ∈ {3,5,8}
Goal: show **QIGOA dominates** on non-extensive entropy landscape.

In [ ]:
cfg_C = ExpConfig(pop_size=50, max_iter=150, n_runs=10,
                  k_levels=(3, 5, 8), fitness='tsallis_q=0.5', algos=ALGOS)

all_rows_C = []
t0 = _t.time()
for i, s in enumerate(slices):
    rows = run_one(s.image, s.mask, s.image_id, cfg_C, base_seed=i*1000 + 9000)
    all_rows_C.extend(rows)
    pd.DataFrame(all_rows_C).to_csv(OUT / 'raw_C_tsallis.csv', index=False)
    eta = (_t.time()-t0)/(i+1) * (len(slices)-i-1)
    print(f'C [{i+1}/{len(slices)}] {s.image_id}  ETA {eta/60:.1f}min', flush=True)

df_C = pd.DataFrame(all_rows_C)
print(f'\n✅ Phase C DONE — {len(df_C)} rows, {(_t.time()-t0)/60:.1f} min')

## 7. Aggregate + bảng tóm tắt

In [ ]:
def summarize(df, phase_name):
    agg = aggregate(df)
    s = agg.groupby(['algorithm', 'k']).agg(
        fitness=('fit_mean', 'mean'), fit_std=('fit_std', 'mean'),
        psnr=('psnr_mean', 'mean'), ssim=('ssim_mean', 'mean'),
        dice=('dice_mean', 'mean'), iou=('iou_mean', 'mean'),
        time=('time_mean', 'mean'),
    ).reset_index()
    s.to_csv(OUT / f'summary_{phase_name}.csv', index=False)
    return agg, s

agg_A, sum_A = summarize(df_A, 'A_kapur_standard')
agg_B, sum_B = summarize(df_B, 'B_kapur_highk')
agg_C, sum_C = summarize(df_C, 'C_tsallis')

print('=== Phase A: Kapur k=2..5 (fitness mean) ===')
print(sum_A.pivot_table(index='algorithm', columns='k', values='fitness').round(4))
print('\n=== Phase B: Kapur k=6,8,10 (fitness mean) ===')
print(sum_B.pivot_table(index='algorithm', columns='k', values='fitness').round(4))
print('\n=== Phase C: Tsallis k=3,5,8 (fitness mean) ===')
print(sum_C.pivot_table(index='algorithm', columns='k', values='fitness').round(4))
print('\n=== Phase B Dice ===')
print(sum_B.pivot_table(index='algorithm', columns='k', values='dice').round(4))

## 8. Statistical tests (Wilcoxon + Friedman + Holm) per phase

In [ ]:
def run_stats(agg, ks, tag):
    rows_w = []
    rows_f = []
    for k in ks:
        sub = agg[agg.k == k]
        pivot = sub.pivot(index='image_id', columns='algorithm', values='fit_best').dropna()
        if 'QIGOA' in pivot.columns:
            q = pivot['QIGOA'].values
            base = {c: pivot[c].values for c in pivot.columns if c != 'QIGOA'}
            wlx = wilcoxon_vs_baselines(q, base, alternative='greater')
            for name, v in wlx.items():
                rows_w.append({'phase': tag, 'k': k, 'baseline': name, **v})
        scores = {c: pivot[c].values for c in pivot.columns}
        stat, p, holm = friedman_with_holm(scores)
        rows_f.append({'phase': tag, 'k': k, 'friedman_stat': stat, 'p_value': p,
                       **{f'holm_p_{n}': v for n, v in holm.items()}})
    return pd.DataFrame(rows_w), pd.DataFrame(rows_f)

wA, fA = run_stats(agg_A, cfg_A.k_levels, 'A')
wB, fB = run_stats(agg_B, cfg_B.k_levels, 'B')
wC, fC = run_stats(agg_C, cfg_C.k_levels, 'C')
wilcoxon_all = pd.concat([wA, wB, wC], ignore_index=True)
friedman_all = pd.concat([fA, fB, fC], ignore_index=True)
wilcoxon_all.to_csv(OUT / 'wilcoxon_all.csv', index=False)
friedman_all.to_csv(OUT / 'friedman_all.csv', index=False)

print('=== Wilcoxon (QIGOA > each baseline) ===')
print(wilcoxon_all[wilcoxon_all.significant == True])

## 9. Convergence figure (k=4 Kapur)

In [ ]:
rep = slices[0]
f_obj, _ = make_kapur_objective(rep.image)
histories = {}
for name in ALGOS:
    opt = REGISTRY[name](dim=4, lb=1, ub=254, pop_size=50, max_iter=150, seed=7)
    res = opt.optimize(f_obj)
    histories[name] = res.history

plt.figure(figsize=(8, 4.5))
colors = {'GA':'tab:blue','PSO':'tab:orange','GOA':'tab:green','WOA':'tab:red',
          'GWO':'tab:purple','MPA':'tab:brown','QIGOA':'magenta'}
for name, h in histories.items():
    is_q = name == 'QIGOA'
    plt.plot(h, label=name, linewidth=2.5 if is_q else 1.2,
             linestyle='-' if is_q else '--', color=colors.get(name))
plt.xlabel('Iteration'); plt.ylabel('Best Kapur entropy')
plt.title(f'Convergence on {rep.image_id} (k=4)')
plt.legend(ncol=2, fontsize=9); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(OUT / 'convergence_k4.png', dpi=200, bbox_inches='tight'); plt.show()

## 10. Visual comparison (k=4)

In [ ]:
rep = slices[0]; k_show = 4
f_obj, _ = make_kapur_objective(rep.image)
fig, axes = plt.subplots(2, 4, figsize=(15, 7.5))
axes[0,0].imshow(rep.image, cmap='gray'); axes[0,0].set_title('Original FLAIR'); axes[0,0].axis('off')
axes[0,1].imshow(rep.mask, cmap='Reds'); axes[0,1].set_title('GT mask'); axes[0,1].axis('off')
for i, name in enumerate(['GOA','WOA','GWO','MPA','GA','QIGOA']):
    r, c = divmod(i+2, 4)
    opt = REGISTRY[name](dim=k_show, lb=1, ub=254, pop_size=50, max_iter=150, seed=11)
    res = opt.optimize(f_obj)
    seg = apply_thresholds(rep.image, res.best_x)
    pred = segmentation_to_binary(seg)
    m = all_metrics(rep.image, rep.image, rep.mask, pred)
    axes[r,c].imshow(seg, cmap='nipy_spectral'); axes[r,c].axis('off')
    axes[r,c].set_title(f"{name}  Dice={m['Dice']:.3f}", fontsize=10,
                         color='red' if name=='QIGOA' else 'black')
plt.tight_layout(); plt.savefig(OUT / 'visual_compare.png', dpi=200, bbox_inches='tight'); plt.show()

## 11. Scaling-with-k figure (paper's KEY figure for narrative)

In [ ]:
# Gộp Phase A + B để vẽ scaling QIGOA vs baselines theo k
df_AB = pd.concat([df_A, df_B], ignore_index=True)
scaling = df_AB.groupby(['algorithm', 'k'])['fitness_value'].mean().reset_index()
scaling.to_csv(OUT / 'scaling_with_k.csv', index=False)

plt.figure(figsize=(8, 5))
for name in ALGOS:
    sub = scaling[scaling.algorithm == name].sort_values('k')
    is_q = name == 'QIGOA'
    plt.plot(sub.k, sub.fitness_value, marker='o' if is_q else 's',
             label=name, linewidth=2.5 if is_q else 1.2,
             markersize=10 if is_q else 6,
             color=colors.get(name))
plt.xlabel('Number of thresholds (k)'); plt.ylabel('Mean Kapur fitness')
plt.title('Scaling with k — QIGOA dominates at high dimensionality')
plt.legend(ncol=2, fontsize=9); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(OUT / 'scaling_with_k.png', dpi=200, bbox_inches='tight'); plt.show()

## 12. Boxplot Dice across phases

In [ ]:
df_all = pd.concat([df_A.assign(phase='Kapur k=2-5'),
                    df_B.assign(phase='Kapur k=6,8,10'),
                    df_C.assign(phase='Tsallis k=3,5,8')], ignore_index=True)
df_all.to_csv(OUT / 'raw_all.csv', index=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for ax, (phase, sub) in zip(axes, df_all.groupby('phase')):
    order = list(ALGOS)
    data = [sub[sub.algorithm == a]['Dice'].values for a in order]
    bp = ax.boxplot(data, labels=order, patch_artist=True)
    for patch, name in zip(bp['boxes'], order):
        patch.set_facecolor('magenta' if name == 'QIGOA' else 'lightgray')
    ax.set_title(phase); ax.grid(alpha=0.3, axis='y')
    ax.tick_params(axis='x', rotation=30)
axes[0].set_ylabel('Dice')
plt.suptitle('Dice distribution across all phases', y=1.02)
plt.tight_layout(); plt.savefig(OUT / 'dice_boxplot.png', dpi=200, bbox_inches='tight'); plt.show()

In [ ]:
print('=== Files in', OUT, '===')
for f in sorted(os.listdir(OUT)):
    sz = os.path.getsize(OUT / f) / 1024
    print(f'  {f:<35s}  {sz:8.1f} KB')

## Download về máy
Tất cả file ở `/kaggle/working/results_v2/`. Copy vào `d:/HoangLong/Personal/paper/3-/qigoa/results_v2/` rồi báo lại để viết paper.